In [30]:
import numpy as np
import pandas as pd
from collections import defaultdict
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
from Bio.PDB.Polypeptide import is_aa


def extract_labeled_coordinates(pdb_file: str, peptide_seq: str, cdr_sequences: dict, mhc_sequence: str):
    """
    Extracts coordinates for all atoms in TCR CDRs, MHC and peptide.
    seq_dict is built from the passed cdr_sequences + MHC + peptide.
    """
    seq_dict = {**cdr_sequences, mhc_sequence: "MHC", peptide_seq: "peptide"}

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("complex", pdb_file)

    coords_all = {}
    atom_to_ca = {}

    for model in structure:
        for chain in model.get_chains():
            residues = [res for res in chain.get_residues() if is_aa(res, standard=True)]
            chain_seq = "".join(seq1(res.get_resname()) for res in residues)

            for target_seq, label in seq_dict.items():
                if target_seq not in chain_seq:
                    continue

                start_idx = chain_seq.find(target_seq)
                matched_residues = residues[start_idx : start_idx + len(target_seq)]

                for residue in matched_residues:
                    aa_1 = seq1(residue.get_resname())
                    res_num = residue.get_id()[1]
                    ca_key = (label, aa_1, res_num)

                    for atom in residue.get_atoms():
                        atom_name = atom.get_name()
                        atom_key = (*ca_key, atom_name)
                        coords_all[atom_key] = atom.get_coord().tolist()
                        atom_to_ca[atom_key] = ca_key

    return coords_all, atom_to_ca


def find_interchain_atomic_contacts(coords_all: dict, atom_to_ca: dict, max_distance: float = 5.0):
    """Returns atomic contacts only between different groups (peptide–TCR, CDR–MHC, etc.)."""
    contacts = []
    keys = list(coords_all.keys())

    for i in range(len(keys)):
        key_i = keys[i]
        ca_i = atom_to_ca.get(key_i)
        if ca_i is None:
            continue
        coord_i = np.array(coords_all[key_i])

        for j in range(i + 1, len(keys)):
            key_j = keys[j]
            ca_j = atom_to_ca.get(key_j)
            if ca_j is None or ca_i == ca_j or ca_i[0] == ca_j[0]:
                continue

            dist = np.linalg.norm(coord_i - np.array(coords_all[key_j]))
            if dist <= max_distance:
                contacts.append((key_i, key_j))

    return contacts


def get_peptide_contacts_df(atomic_contacts, peptide_seq: str, pdb_path: str):
    """Amino-acid contact matrix for the peptide (one row per position)."""
    pep_contacts = [c for c in atomic_contacts if c[0][0] == "peptide" or c[1][0] == "peptide"]
    res_contacts = defaultdict(set)

    for at1, at2 in pep_contacts:
        if at1[0] == "peptide":
            pep_key = at1[:3]
            partner = at2[:3]
        else:
            pep_key = at2[:3]
            partner = at1[:3]
        res_contacts[pep_key].add(partner)

    rows = []
    for i, aa in enumerate(peptide_seq, start=1):
        pep_key = ("peptide", aa, i)
        contacts_set = res_contacts.get(pep_key, set())
        rows.append({
            "peptide_seq": peptide_seq,
            "pdb_path": pdb_path,
            "pep_pos": i,
            "pep_aa": aa,
            "contacts": contacts_set,
            "n_contacts": len(contacts_set),
        })

    return pd.DataFrame(rows)


def get_tcr_mhc_contacts_df(atomic_contacts, peptide_seq: str, pdb_path: str):
    """TCR (any CDR) ↔ MHC amino-acid contacts."""
    tcr_mhc_pairs = []
    for at1, at2 in atomic_contacts:
        label1, label2 = at1[0], at2[0]
        if ("CDR" in label1 or "CDR" in label2) and ("MHC" in label1 or "MHC" in label2):
            if "CDR" in label1:
                tcr_mhc_pairs.append((at1[:3], at2[:3]))
            else:
                tcr_mhc_pairs.append((at2[:3], at1[:3]))

    unique_pairs = {tuple(sorted([a, b])) for a, b in tcr_mhc_pairs}

    rows = []
    for tcr_res, mhc_res in unique_pairs:
        rows.append({
            "peptide_seq": peptide_seq,
            "pdb_path": pdb_path,
            "tcr_label": tcr_res[0],
            "tcr_aa": tcr_res[1],
            "tcr_num": tcr_res[2],
            "mhc_aa": mhc_res[1],
            "mhc_num": mhc_res[2],
        })

    return pd.DataFrame(rows)


def process_pdb(pdb_path: str,
                peptide_seq: str,
                cdr_sequences: dict,
                mhc_sequence: str,
                max_distance: float = 5.0):
    """
    Main function.
    You now pass cdr_sequences (sequence → label) and mhc_sequence as arguments.
    """
    coords_all, atom_to_ca = extract_labeled_coordinates(pdb_path, peptide_seq, cdr_sequences, mhc_sequence)
    atomic_contacts = find_interchain_atomic_contacts(coords_all, atom_to_ca, max_distance)

    peptide_df = get_peptide_contacts_df(atomic_contacts, peptide_seq, pdb_path)
    tcr_mhc_df = get_tcr_mhc_contacts_df(atomic_contacts, peptide_seq, pdb_path)

    return peptide_df, tcr_mhc_df

In [31]:
# Define your sequences once (outside the function)
cdr_sequences = {
    "DRGSQS":          "CDR1_alpha",
    "IYSNGD":          "CDR2_alpha",
    "CAVNVAGKSTF":     "CDR3_alpha",
    "GTSNPN":          "CDR1_beta",
    "SVGIG":           "CDR2_beta",
    "CAWSETGLGTGELFF": "CDR3_beta",
}

mhc_sequence = (
    "GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLRGYYNQSEAGSHTVQRMYGCDVGSDWRFLRGYHQYAYDGKDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQ"
)

# Example for one PDB
pdb_file = "/projects/structures/peptide_swap_mel5/FLTGLGIVVI/FLTGLGIVVI_pmhc_oc/ranked_0.pdb"
peptide  = "FLTGLGIVVI"

pep_contacts, tcr_mhc_contacts = process_pdb(
    pdb_path=pdb_file,
    peptide_seq=peptide,
    cdr_sequences=cdr_sequences,
    mhc_sequence=mhc_sequence,
    max_distance=5.0
)

display(pep_contacts)
display(tcr_mhc_contacts.head())

,peptide_seq,pdb_path,pep_pos,pep_aa,contacts,n_contacts
0,FLTGLGIVVI,/projects/structures/peptide_swap_mel5/FLTGLGI...,1,F,"{(CDR1_alpha, Q, 31), (MHC, Y, 7), (MHC, F, 33...",12
1,FLTGLGIVVI,/projects/structures/peptide_swap_mel5/FLTGLGI...,2,L,"{(MHC, H, 70), (CDR1_alpha, Q, 31), (MHC, Y, 7...",10
2,FLTGLGIVVI,/projects/structures/peptide_swap_mel5/FLTGLGI...,3,T,"{(MHC, L, 156), (MHC, H, 70), (CDR1_alpha, Q, ...",7
3,FLTGLGIVVI,/projects/structures/peptide_swap_mel5/FLTGLGI...,4,G,"{(CDR1_alpha, Q, 31), (CDR3_beta, L, 98), (MHC...",6
4,FLTGLGIVVI,/projects/structures/peptide_swap_mel5/FLTGLGI...,5,L,"{(MHC, L, 156), (CDR1_alpha, Q, 31), (CDR3_bet...",8
5,FLTGLGIVVI,/projects/structures/peptide_swap_mel5/FLTGLGI...,6,G,"{(MHC, L, 156), (MHC, V, 152), (MHC, H, 114), ...",7
6,FLTGLGIVVI,/projects/structures/peptide_swap_mel5/FLTGLGI...,7,I,"{(MHC, L, 156), (MHC, H, 70), (MHC, T, 73), (M...",10
7,FLTGLGIVVI,/projects/structures/peptide_swap_mel5/FLTGLGI...,8,V,"{(MHC, D, 77), (MHC, T, 73), (CDR3_beta, G, 97...",9
8,FLTGLGIVVI,/projects/structures/peptide_swap_mel5/FLTGLGI...,9,V,"{(MHC, D, 77), (MHC, T, 73), (CDR3_beta, T, 96...",6
9,FLTGLGIVVI,/projects/structures/peptide_swap_mel5/FLTGLGI...,10,I,"{(MHC, T, 80), (MHC, Y, 123), (MHC, D, 77), (M...",9


,peptide_seq,pdb_path,tcr_label,tcr_aa,tcr_num,mhc_aa,mhc_num
0,FLTGLGIVVI,/projects/structures/peptide_swap_mel5/FLTGLGI...,CDR2_beta,V,51,Q,72
1,FLTGLGIVVI,/projects/structures/peptide_swap_mel5/FLTGLGI...,CDR2_beta,I,53,R,75
2,FLTGLGIVVI,/projects/structures/peptide_swap_mel5/FLTGLGI...,CDR2_beta,V,51,K,68
3,FLTGLGIVVI,/projects/structures/peptide_swap_mel5/FLTGLGI...,CDR3_beta,T,100,Q,155
4,FLTGLGIVVI,/projects/structures/peptide_swap_mel5/FLTGLGI...,CDR1_alpha,S,30,T,163


# Contacts counts

In [32]:
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
from Bio.PDB.Polypeptide import is_aa


def get_chain_sequence(pdb_path: str, chain_id: str) -> str:
    """
    Returns the 1-letter amino acid sequence of a specific chain in a PDB file.
    
    Parameters:
        pdb_path (str): Path to the PDB file
        chain_id (str): Chain identifier (e.g. 'A', 'B', 'C', 'D')
    
    Returns:
        str: Amino acid sequence (1-letter code). Empty string if chain not found.
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("structure", pdb_path)

    for model in structure:
        for chain in model.get_chains():
            if chain.get_id() == chain_id:
                # Build sequence only from standard amino acids
                residues = [res for res in chain.get_residues() if is_aa(res, standard=True)]
                sequence = "".join(seq1(res.get_resname()) for res in residues)
                return sequence

    return ""

In [35]:
get_chain_sequence('/projects/structures/peptide_swap_mel5/FLTGLGIVVI/FLTGLGIVVI_pmhc_oc/ranked_0.pdb', 'C')

'FLTGLGIVVI'

In [36]:
import numpy as np
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa


def count_tcr_peptide_aa_contacts(pdb_path: str, max_distance: float = 5.0) -> int:
    """
    EFFICIENT version.
    Returns the TOTAL NUMBER OF AMINO-ACID CONTACTS
    (unique residue–residue pairs) between TCR (chains A + B) and peptide (chain C).

    Two residues are considered in contact if ANY of their atoms are within max_distance Å.
    
    Only argument needed: pdb_path.
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("complex", pdb_path)

    # Pre-collect residues + their atom coordinates as numpy arrays
    tcr_residues = []      # list of (chain_id, resnum, atom_coords_array)
    peptide_residues = []  # list of (resnum, atom_coords_array)

    for model in structure:
        for chain in model.get_chains():
            chain_id = chain.get_id()

            for residue in chain.get_residues():
                if not is_aa(residue, standard=True):
                    continue

                # Collect coordinates of ALL atoms in this residue
                atom_coords = [atom.get_coord() for atom in residue.get_atoms()]
                if not atom_coords:
                    continue

                coords_array = np.array(atom_coords)          # shape: (n_atoms, 3)
                resnum = residue.get_id()[1]

                if chain_id in ('A', 'B'):                    # TCR
                    tcr_residues.append((chain_id, resnum, coords_array))
                elif chain_id == 'C':                         # Peptide
                    peptide_residues.append((resnum, coords_array))

    if not tcr_residues or not peptide_residues:
        return 0

    contact_set = set()

    # Vectorized distance check for every TCR–peptide residue pair
    for tcr_chain, tcr_num, tcr_atoms in tcr_residues:
        for pep_num, pep_atoms in peptide_residues:
            # Broadcast: compute all atom–atom distances between the two residues
            diff = tcr_atoms[:, np.newaxis, :] - pep_atoms[np.newaxis, :, :]
            distances = np.linalg.norm(diff, axis=-1)          # shape: (n_tcr_atoms, n_pep_atoms)

            if np.any(distances <= max_distance):
                contact_set.add((tcr_chain, tcr_num, pep_num))

    return len(contact_set)

In [38]:
# Single example
pdb_file = "/projects/structures/peptide_swap_1e6/CIFGPDFPVI/CIFGPDFPVI_pmhc_oc/ranked_0.pdb"
num_contacts = count_tcr_peptide_aa_contacts(pdb_file)
print(f"Total TCR–peptide amino-acid contacts: {num_contacts}")

Total TCR–peptide amino-acid contacts: 17


# Contacts statictics

In [44]:
import tqdm

structures = os.listdir('/projects/structures/clusters/HLA-A/')

res_dict_a = {}

for structure in tqdm.tqdm(structures):
    
    structure_path = f'/projects/structures/clusters/HLA-A/{structure}/{structure}_pmhc_oc/ranked_0.pdb'
    
    if os.path.exists(structure_path):
        
        num_contacts = count_tcr_peptide_aa_contacts(structure_path)
        
        res_dict_a[structure] = num_contacts

100%|██████████| 9435/9435 [33:25<00:00,  4.70it/s]   


In [45]:
structures = os.listdir('/projects/structures/clusters/HLA-B/')

res_dict_b = {}

for structure in tqdm.tqdm(structures):
    
    structure_path = f'/projects/structures/clusters/HLA-B/{structure}/{structure}_pmhc_oc/ranked_0.pdb'
    
    if os.path.exists(structure_path):
        
        num_contacts = count_tcr_peptide_aa_contacts(structure_path)
        
        res_dict_b[structure] = num_contacts

100%|██████████| 396/396 [01:18<00:00,  5.06it/s]


In [47]:
structures = os.listdir('/projects/structures/clusters/HLA-E/')

res_dict_e = {}

for structure in tqdm.tqdm(structures):
    
    structure_path = f'/projects/structures/clusters/HLA-E/{structure}/{structure}_pmhc_oc/ranked_0.pdb'
    
    if os.path.exists(structure_path):
        
        num_contacts = count_tcr_peptide_aa_contacts(structure_path)
        
        res_dict_e[structure] = num_contacts

100%|██████████| 9/9 [00:01<00:00,  4.55it/s]


In [49]:
res_contacts = pd.Series(res_dict_e | res_dict_b | res_dict_a).T
res_contacts

fb95e51bc7d7e699457002ecb3cbbc9cce7b1628c3785096f3cbb920d41450eb     8
5c48c8042cd6405830593d72e1ac3498a24a15a3a4d1d6b258d6add7f597f6bd    13
4c55d51c798c017d37736929b26903c977a070204f97bef91559c9053f997217    18
b765d2c65831ba7d99fc49145b7291f56e0f8ca8d4f678c2b8b539be95861356    20
30545cbd3cc0a4290efdbd757c43d9bc20adce5a1a77c17ad27cfda05885b9fb    17
                                                                    ..
8d758d286487ce18fd3b2b3d23d796aac6d25d0f4ab926bb19c0c0e80e730d2b    12
c59816afa7843b3ae2093d064da816d743ce973523904f5fd98cff916c92d17c    19
f4a15f4197af652eb6a3a8f7557d9776305e3856fecf7441fc68db0116e9d236    18
f78501d4a54031b2a4eacc2b30e8927fea181f42535650ff745ccc1dbe78a251    22
0ddac88ccd4fb830fd5463344820599395114ca8c18a1e0d976c635eae664d19    28
Length: 9804, dtype: int64

In [50]:
res_contacts.to_csv('contacts_curr_structures.tsv', sep='\t')